# 8. Pace & outcomes: entry type

Compare **Controlled and Dumped 5v5 entries** by total possession pace before the first entry.

## 8.1. Load inputs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROCESSED_DATA_DIR = Path("../data/processed")

# Event IDs preserve order, including events sharing a clock value.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
entry_candidates = events.loc[
    events["event"].eq("Zone Entry")
    & events["is_5v5"]
    & events["detail_1"].isin(["Carried", "Played", "Dumped"])
].copy()
# Use each possession's first qualifying entry, regardless of its starting event.
entry_attempts = entry_candidates.groupby("possession_id", sort=False).head(1).copy()

## 8.2. Calculate preceding total pace

Pool same-possession movements through the approach ending at the entry.

In [2]:
def add_entry_pace(entry_events, transition_events):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for entry in entry_events.itertuples(index=False):
        history = movement_groups.get(entry.possession_id, transition_events.iloc[:0])
        # Include arrival at the entry release, but never subsequent movement.
        history = history.loc[history["end_event_id"].le(entry.event_id)]
        distance = float(history["distance_ft"].sum())
        duration = float(history["modeled_elapsed_seconds"].sum())
        rows.append(
            {
                "event_id": entry.event_id,
                "preceding_transitions": len(history),
                "preceding_distance_ft": distance,
                "preceding_movement_seconds": duration,
                "preceding_total_pace_ft_s": distance / duration
                if duration > 0
                else np.nan,
            }
        )
    return entry_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


entry_attempts = add_entry_pace(entry_attempts, movements)

## 8.3. Identify entry outcomes

Carried and Played entries are **Controlled**; Dumped entries are **Dumped**. Possessions lost before entry are outside the denominator.

In [3]:
entry_attempts["entry_outcome"] = np.where(
    entry_attempts["detail_1"].isin(["Carried", "Played"]), "Controlled", "Dumped"
)
entry_attempts["controlled_entry"] = entry_attempts["entry_outcome"].eq("Controlled")

## 8.4. Summarize quartiles

In [4]:
def summarize_entry_types(data):
    eligible = data.loc[data["preceding_total_pace_ft_s"].notna()].copy()
    bands, edges = pd.qcut(
        eligible["preceding_total_pace_ft_s"],
        q=4,
        labels=False,
        retbins=True,
        duplicates="drop",
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    summary = (
        eligible.loc[eligible["quartile"].notna()]
        .groupby("quartile", observed=True)
        .agg(
            entries=("event_id", "size"),
            controlled_entries=("controlled_entry", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=("preceding_total_pace_ft_s", "median"),
        )
        .reset_index()
    )
    summary["dumped_entries"] = summary["entries"] - summary["controlled_entries"]
    summary["controlled_entry_pct"] = (
        100 * summary["controlled_entries"] / summary["entries"]
    )
    summary["dumped_entry_pct"] = 100 * summary["dumped_entries"] / summary["entries"]
    return summary.merge(boundaries, on="quartile", validate="one_to_one"), boundaries


entry_type_summary, quartile_boundaries = summarize_entry_types(entry_attempts)
entry_coverage = (
    entry_attempts.assign(
        missing_pace=entry_attempts["preceding_total_pace_ft_s"].isna(),
        zero_pace=entry_attempts["preceding_total_pace_ft_s"].eq(0),
        measurable=entry_attempts["preceding_total_pace_ft_s"].notna(),
    )
    .groupby("entry_outcome")
    .agg(
        entries=("event_id", "size"),
        missing_pace=("missing_pace", "sum"),
        zero_pace=("zero_pace", "sum"),
        measurable=("measurable", "sum"),
    )
    .reset_index()
)
print(f"Excluded {len(entry_candidates) - len(entry_attempts)} repeat entries.")
display(entry_coverage)

Excluded 37 repeat entries.


,entry_outcome,entries,missing_pace,zero_pace,measurable
0,Controlled,2355,16,0,2339
1,Dumped,1259,59,15,1200


## 8.5. Display entry outcomes

Q1 is slowest and Q4 fastest. Each stacked bar shows Controlled and Dumped shares totaling 100%. Hover shows counts and pace boundaries; empty groups stay blank.

In [5]:
def entry_type_figure(summary, boundaries):
    outcomes = [
        ("controlled_entry_pct", "Controlled", "#2563eb", "white"),
        ("dumped_entry_pct", "Dumped", "#e2e8f0", "#334155"),
    ]
    figure = go.Figure()
    bars = summary.set_index("quartile").reindex(range(1, len(boundaries) + 1))
    customdata = bars[
        [
            "entries",
            "controlled_entries",
            "dumped_entries",
            "games",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
            "median_pace_ft_s",
        ]
    ].to_numpy()
    for metric, label, color, text_color in outcomes:
        figure.add_trace(
            go.Bar(
                x=[f"Q{q}" for q in bars.index],
                y=bars[metric],
                name=label,
                marker_color=color,
                insidetextfont=dict(color=text_color),
                text=[
                    f"{value:.1f}%" if pd.notna(value) else "" for value in bars[metric]
                ],
                textposition="inside",
                insidetextanchor="middle",
                customdata=customdata,
                hovertemplate=(
                    f"%{{x}}<br>{label}: %{{y:.1f}}%"
                    "<br>Entries: %{customdata[0]:.0f}"
                    "<br>Controlled entries: %{customdata[1]:.0f}"
                    "<br>Dumped entries: %{customdata[2]:.0f}"
                    "<br>Games: %{customdata[3]:.0f}"
                    "<br>Pace range: %{customdata[4]:.2f} to %{customdata[5]:.2f} ft/s"
                    "<br>Median pace: %{customdata[6]:.1f} ft/s<extra></extra>"
                ),
            ),
        )
    if summary.empty:
        note = "No data: no measurable pace with assigned bands."
    elif len(boundaries) < 4:
        note = f"Only {len(boundaries)} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title="Total pace before entry | 5v5 | first entry per possession",
        template="plotly_white",
        barmode="stack",
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.05),
        height=500,
        margin=dict(t=100, b=100),
    )
    figure.update_xaxes(
        title_text="Preceding total possession pace quartile",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    figure.update_yaxes(range=[0, 100], title_text="Entries (%)")
    return figure


entry_comparison_figure = entry_type_figure(entry_type_summary, quartile_boundaries)
entry_comparison_figure.show()

## 8.6. Export entry-type outcomes

In [6]:
entry_type_outcomes_df = (
    entry_type_summary[
        [
            "quartile",
            "entries",
            "controlled_entries",
            "dumped_entries",
            "games",
            "median_pace_ft_s",
            "controlled_entry_pct",
            "dumped_entry_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "quartile": "Int64",
            "entries": "Int64",
            "controlled_entries": "Int64",
            "dumped_entries": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "controlled_entry_pct": "Float64",
            "dumped_entry_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values("quartile")
    .reset_index(drop=True)
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_path = PROCESSED_DATA_DIR / "entry_type_outcomes.parquet"
entry_type_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, entry_type_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 4 rows and 10 columns to ..\data\processed\entry_type_outcomes.parquet


,quartile,entries,controlled_entries,dumped_entries,games,median_pace_ft_s,controlled_entry_pct,dumped_entry_pct,lower_pace_ft_s,upper_pace_ft_s
0,1,885,508,377,34,12.747549,57.40113,42.59887,0.0,15.962143
1,2,885,561,324,34,18.532529,63.389831,36.610169,15.962143,20.524376
2,3,884,620,264,34,22.543256,70.135747,29.864253,20.524376,25.350943
3,4,885,650,235,34,29.493353,73.446328,26.553672,25.350943,198.408373
